## 📊 RQ4

How LLM perform on real world apps?

In [1]:
# Imports
from   dotenv          		import load_dotenv
from   matplotlib      		import colors as mcolors
import matplotlib.pyplot 	as plt
import pandas          		as pd
import datetime
import json
import os

##### Initialization


In [2]:
print("⚡ START: {} ⚡".format(datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")))
initTime = datetime.datetime.now()

⚡ START: 2026-05-30 17:30:22 ⚡


In [3]:
# Load .env Info
load_dotenv()

True

#### 📥 1] Load Data


Parameters

In [ ]:
# Data Path
DATA_PATH            = "../../0_Data/androzoo.csv"

# Output Paths
OUTPUT_DIR           = "./Data"
FILTERED_OUTPUT_PATH = os.path.join(OUTPUT_DIR, "filteredApps.csv")
SAMPLE_OUTPUT_PATH   = os.path.join(OUTPUT_DIR, "sampledApps.csv")
CHUNK_SIZE           = 250_000

os.makedirs(OUTPUT_DIR, exist_ok = True)

#### 🔎 2] Filter 2026 Google Play Apps


In [5]:
filteredChunks = []
processedRows   = 0
matchedRows     = 0

for chunkIndex, chunkDf in enumerate(pd.read_csv(DATA_PATH, chunksize = CHUNK_SIZE, low_memory = False), start = 1):
	processedRows += len(chunkDf)
	addedIn2026Mask = chunkDf["added"].astype("string").str.startswith("2026-", na = False)
	googlePlayMask  = chunkDf["markets"].astype("string").str.contains("play.google.com", case = False, na = False, regex = False)
	filteredChunkDf = chunkDf[addedIn2026Mask & googlePlayMask].copy()

	if not filteredChunkDf.empty:
		filteredChunks.append(filteredChunkDf)
		matchedRows += len(filteredChunkDf)

	if chunkIndex % 10 == 0:
		print("🔄 Processed {:,} rows | matched {:,} apps".format(processedRows, matchedRows))

if len(filteredChunks) == 0:
	raise ValueError("No Google Play apps added in 2026 were found in {}".format(DATA_PATH))

filteredAppsDf = pd.concat(filteredChunks, ignore_index = True)
filteredOutputDf = filteredAppsDf[["sha256", "pkg_name", "added", "vt_detection"]].copy()
filteredOutputDf.columns = ["sha256", "pkgName", "addedDate", "vtScore"]
filteredOutputDf.to_csv(FILTERED_OUTPUT_PATH, index = False)

print("\n✅ Filtered {:,} Google Play apps added in 2026".format(len(filteredAppsDf)))
print("💾 Saved filtered apps to {}".format(FILTERED_OUTPUT_PATH))

🔄 Processed 2,500,000 rows | matched 47,330 apps
🔄 Processed 5,000,000 rows | matched 94,253 apps
🔄 Processed 7,500,000 rows | matched 141,409 apps
🔄 Processed 10,000,000 rows | matched 188,635 apps
🔄 Processed 12,500,000 rows | matched 235,718 apps
🔄 Processed 15,000,000 rows | matched 282,674 apps
🔄 Processed 17,500,000 rows | matched 330,239 apps
🔄 Processed 20,000,000 rows | matched 377,944 apps
🔄 Processed 22,500,000 rows | matched 425,106 apps
🔄 Processed 25,000,000 rows | matched 472,528 apps
🔄 Processed 27,271,540 rows | matched 515,530 apps

✅ Filtered 515,530 Google Play apps added in 2026
💾 Saved filtered apps to Data/filteredApps.csv


#### 📊 3] Stats


In [6]:
apkSizeMb = pd.to_numeric(filteredAppsDf["apk_size"], errors = "coerce") / (1024 * 1024)
vtScores  = pd.to_numeric(filteredAppsDf["vt_detection"], errors = "coerce")
addedDates = pd.to_datetime(filteredAppsDf["added"], errors = "coerce")

print("📦 Apps                   : {:,}".format(len(filteredAppsDf)))
print("🧩 Unique packages         : {:,}".format(filteredAppsDf["pkg_name"].nunique()))
print("📅 Added date range        : {} → {}".format(addedDates.min(), addedDates.max()))
print("💾 APK size total          : {:,.2f} GB".format(apkSizeMb.sum() / 1024))
print("📏 APK size MB avg / median: {:,.2f} / {:,.2f}".format(apkSizeMb.mean(), apkSizeMb.median()))
print("📐 APK size MB min / max   : {:,.2f} / {:,.2f}".format(apkSizeMb.min(), apkSizeMb.max()))
print("🛡️ VT score avg / median   : {:,.2f} / {:,.2f}".format(vtScores.mean(), vtScores.median()))
print("🚨 VT score min / max      : {:,.0f} / {:,.0f}".format(vtScores.min(), vtScores.max()))
print("✅ VT score zero           : {:,} ({:.2%})".format((vtScores == 0).sum(), (vtScores == 0).mean()))
print("⚠️ VT score positive       : {:,} ({:.2%})".format((vtScores > 0).sum(), (vtScores > 0).mean()))
print("❓ VT score missing        : {:,} ({:.2%})".format(vtScores.isna().sum(), vtScores.isna().mean()))

📦 Apps                   : 515,530
🧩 Unique packages         : 316,241
📅 Added date range        : 2026-01-01 00:00:03.176667 → 2026-05-30 05:35:58.020951
💾 APK size total          : 20,167.43 GB
📏 APK size MB avg / median: 40.06 / 27.03
📐 APK size MB min / max   : 0.01 / 1,459.12
🛡️ VT score avg / median   : 0.02 / 0.00
🚨 VT score min / max      : 0 / 22
✅ VT score zero           : 390,187 (75.69%)
⚠️ VT score positive       : 6,537 (1.27%)
❓ VT score missing        : 118,806 (23.05%)


#### 🎲 4] Random Sample


In [ ]:
# Sampling Parameters
RANDOM_SEED = 777
SAMPLE_SIZE = 1_000

In [ ]:
if len(filteredAppsDf) < SAMPLE_SIZE:
	raise ValueError("Cannot sample {:} apps: only {:} matching apps are available".format(SAMPLE_SIZE, len(filteredAppsDf)))

sampleAppsDf = filteredOutputDf.sample(n = SAMPLE_SIZE, random_state = RANDOM_SEED).reset_index(drop = True)
sampleAppsDf.to_csv(SAMPLE_OUTPUT_PATH, index = False)
print("\n🎲 Randomly sampled {:} apps with seed {}".format(len(sampleAppsDf), RANDOM_SEED))
print("💾 Saved sample to {}".format(SAMPLE_OUTPUT_PATH))


🎲 Randomly sampled 1,000 apps with seed 777
💾 Saved sample to Data/sampledApps.csv


##### 🔚 End 

In [ ]:
endTime = datetime.datetime.now()
print("\n🔚 --- END:  {} --- 🔚".format(endTime.strftime("%Y-%m-%d %H:%M:%S")))

totalTime = endTime - initTime
hours = totalTime.total_seconds() // 3600
minutes = (totalTime.total_seconds() % 3600) // 60
print("⏱️ --- Time: {:02d} hours and {:02d} minutes [{:02d} seconds] --- ⏱️".format(int(hours), int(minutes), int(totalTime.total_seconds())))


END:  2026-05-30 17:32:34
Time: 00 hours and 02 minutes [132 seconds]
